# CaboCha で文節単位の係り受けを解析する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_parsing/cabocha.ipynb)

NLP クックブック 14-04 ／ 構文解析

日本語の係り受け解析器として長く使われてきた CaboCha で、文を文節に区切り、どの文節がどの文節に係るかを解析したい。前のレシピの GiNZA の結果とも比べる。

In [1]:
# Colab で 5 分ほど (CRF++ と CaboCha をソースからビルドする。CaboCha の make の中で、IPA・JUMAN・UniDic 用のモデルファイルも作られる)
!apt-get install -y -q automake mecab libmecab-dev mecab-ipadic-utf8 > /dev/null
!git clone -q --depth 1 https://github.com/taku910/crfpp
!git clone -q --depth 1 https://github.com/taku910/cabocha
# CRF++: git 版には winmain.h がないので、それを読み込む行を消してからビルドする
!cd crfpp && sed -i '/winmain.h/d' crf_learn.cpp crf_test.cpp && ./configure -q > /dev/null 2>&1 && make -j4 > /dev/null 2>&1 && make install > /dev/null && ldconfig
# CaboCha: git 版にない install-sh などを automake から補い、文字コードを UTF-8 にしてビルドする
!cd cabocha && cp /usr/share/automake-*/install-sh /usr/share/automake-*/missing /usr/share/automake-*/depcomp . && ./configure -q --with-charset=UTF8 --enable-utf8-only > /dev/null 2>&1 && make -j4 > /dev/null 2>&1 && make install > /dev/null && ldconfig
# Python バインディング: 付属の CaboCha.py は Python 3.12 で廃止された imp を使うので、そこを通らないようにしてから入れる
!sed -i 's/(2,6,0)/(9,0,0)/' cabocha/python/CaboCha.py && pip install -q ./cabocha/python
!pip install -q ginza ja_ginza
!cabocha --version

In [2]:
# 木の表示 (-f0、既定)。各文節の右の D から線をたどった先が係り先
!echo "太郎は図書館で花子に借りた本を昨日返した。" | cabocha -f0
# 格子形式 (-f1)。「* 文節番号 係り先D 主辞/機能語 スコア」の行の下に、MeCab (IPA 辞書) の解析結果が続く
!echo "太郎は図書館で花子に借りた本を昨日返した。" | cabocha -f1

  太郎は-----------D
  図書館で---D     |
      花子に-D     |
        借りた-D   |
            本を---D
              昨日-D
            返した。
EOS
* 0 6D 0/1 -1.474801
太郎	名詞,固有名詞,人名,名,*,*,太郎,タロウ,タロー
は	助詞,係助詞,*,*,*,*,は,ハ,ワ
* 1 3D 0/1 0.737843
図書館	名詞,一般,*,*,*,*,図書館,トショカン,トショカン
で	助詞,格助詞,一般,*,*,*,で,デ,デ
* 2 3D 0/1 1.304699
花子	名詞,固有名詞,人名,名,*,*,花子,ハナコ,ハナコ
に	助詞,格助詞,一般,*,*,*,に,ニ,ニ
* 3 4D 0/1 1.308517
借り	動詞,自立,*,*,一段,連用形,借りる,カリ,カリ
た	助動詞,*,*,*,特殊・タ,基本形,た,タ,タ
* 4 6D 0/1 -1.474801
本	名詞,一般,*,*,*,*,本,ホン,ホン
を	助詞,格助詞,一般,*,*,*,を,ヲ,ヲ
* 5 6D 0/0 -1.474801
昨日	名詞,副詞可能,*,*,*,*,昨日,キノウ,キノー
* 6 -1D 0/1 0.000000
返し	動詞,自立,*,*,五段・サ行,連用形,返す,カエシ,カエシ
た	助動詞,*,*,*,特殊・タ,基本形,た,タ,タ
。	記号,句点,*,*,*,*,。,。,。
EOS


In [3]:
import CaboCha

parser = CaboCha.Parser()  # コマンドと同じオプションを文字列で渡せる (例: CaboCha.Parser("-n 1"))

def cabocha_chunks(text):
    """文節ごとに (文節の文字列, 係り先の文節番号, 主辞の形態素, 機能語の形態素) を返す。係り先 -1 は文末"""
    tree = parser.parse(text)
    out = []
    for i in range(tree.chunk_size()):
        c = tree.chunk(i)
        toks = [tree.token(c.token_pos + j) for j in range(c.token_size)]
        out.append(("".join(t.surface for t in toks), c.link,
                    toks[c.head_pos].surface, toks[c.func_pos].surface))
    return out

text = "太郎は図書館で花子に借りた本を昨日返した。"
chunks = cabocha_chunks(text)
for i, (surface, link, head, func) in enumerate(chunks):
    target = chunks[link][0] if link >= 0 else "(文末)"
    # 「昨日」のように付属語がない文節では、機能語の位置 (func_pos) が主辞と同じになる
    print(f"{i} {surface:　<6} → {target:　<6} 主辞: {head}  機能語: {func}")

0 太郎は　　　 → 返した。　　 主辞: 太郎  機能語: は
1 図書館で　　 → 借りた　　　 主辞: 図書館  機能語: で
2 花子に　　　 → 借りた　　　 主辞: 花子  機能語: に
3 借りた　　　 → 本を　　　　 主辞: 借り  機能語: た
4 本を　　　　 → 返した。　　 主辞: 本  機能語: を
5 昨日　　　　 → 返した。　　 主辞: 昨日  機能語: 昨日
6 返した。　　 → (文末)　　 主辞: 返し  機能語: た


In [4]:
import ginza
import spacy

nlp = spacy.load("ja_ginza")

def ginza_pairs(text):
    """前のレシピと同じ方法で、GiNZA の文節と係り先の組を返す"""
    pairs = []
    for span in ginza.bunsetu_spans(nlp(text)):
        head = ginza.bunsetu_span(span.root.head)
        pairs.append((span.text, "(文末)" if head.start == span.start else head.text))
    return pairs

def cabocha_pairs(text):
    chunks = cabocha_chunks(text)
    return [(s, chunks[link][0] if link >= 0 else "(文末)") for s, link, _, _ in chunks]

for t in [text, "彼は赤い帽子をかぶった女の子に道を教えてあげた。", "吾輩は猫である。名前はまだ無い。"]:
    print(t)
    for name, pairs in [("CaboCha", cabocha_pairs(t)), ("GiNZA", ginza_pairs(t))]:
        print(f"  {name:8}", "  ".join(f"{s}→{h}" for s, h in pairs))

太郎は図書館で花子に借りた本を昨日返した。
  CaboCha  太郎は→返した。  図書館で→借りた  花子に→借りた  借りた→本を  本を→返した。  昨日→返した。  返した。→(文末)
  GiNZA    太郎は→借りた  図書館で→借りた  花子に→借りた  借りた→本を  本を→返した。  昨日→返した。  返した。→(文末)
彼は赤い帽子をかぶった女の子に道を教えてあげた。
  CaboCha  彼は→教えてあげた。  赤い→帽子を  帽子を→かぶった  かぶった→女の子に  女の子に→教えてあげた。  道を→教えてあげた。  教えてあげた。→(文末)
  GiNZA    彼は→あげた。  赤い→帽子を  帽子を→かぶった  かぶった→女の子に  女の子に→教えて  道を→教えて  教えて→あげた。  あげた。→(文末)
吾輩は猫である。名前はまだ無い。
  CaboCha  吾輩は→猫である。  猫である。→無い。  名前は→無い。  まだ→無い。  無い。→(文末)
  GiNZA    吾輩は→猫である。  猫である。→(文末)  名前は→無い。  まだ→無い。  無い。→(文末)


In [5]:
# 固有表現 (-n 1)。IREX の定義の固有表現が、木の表示では <PERSON> などのタグで、形態素ごとには IOB2 形式のタグで得られる
!echo "昨日、山田太郎さんは東京大学でトヨタ自動車の社長と会った。" | cabocha -n 1
ne_parser = CaboCha.Parser("-n 1")  # parse の結果はパーザの中にあるので、パーザを変数に入れておく
ne_tree = ne_parser.parse("昨日、山田太郎さんは東京大学でトヨタ自動車の社長と会った。")
print([(ne_tree.token(i).surface, ne_tree.token(i).ne) for i in range(ne_tree.token_size())
       if ne_tree.token(i).ne != "O"])

                                     昨日、---------D
              <PERSON>山田太郎</PERSON>さんは-------D
        <ORGANIZATION>東京大学</ORGANIZATION>で-----D
      <ORGANIZATION>トヨタ自動車</ORGANIZATION>の-D |
                                             社長と-D
                                             会った。
EOS
[('山田', 'B-PERSON'), ('太郎', 'I-PERSON'), ('東京大学', 'B-ORGANIZATION'), ('トヨタ自動車', 'B-ORGANIZATION')]


- CaboCha は工藤拓（奈良先端大・松本研究室）による、Support Vector Machines (SVM) に基づく日本語係り受け解析器。方式は論文「チャンキングの段階適用による日本語係り受け解析」（Cascaded Chunking、工藤・松本 2002）。付属のモデルでは、係り受けは 2 次の多項式カーネルの SVM、文節の区切りと固有表現は CRF++ の CRF で解析する（CRF++ が必要なのはこのため）。最後のリリースは 0.69（2015 年 1 月）。GitHub (taku910/cabocha) では、2015 年 1 月より後にソースは変わっておらず、説明のページ（index.html など）が追加・修正されただけ。
- 形態素解析には MeCab を使い、品詞体系と辞書（IPA・JUMAN・UniDic）に合ったモデルが必要。ここでは既定の IPA 辞書 (mecab-ipadic-utf8) と IPA モデルを使う。ドキュメントによると、IPA モデルは京都大学テキストコーパス 4.0 の形態素の部分を MeCab で自動解析し直したものから学習していて、コーパスから直接学習した JUMAN モデルより精度が落ちる。
- 付属のモデルは毎日新聞の記事から学習されていて、そのまま使えるのは研究目的・個人利用に限られる（プログラム自体は LGPL か BSD ライセンス）。自分のコーパスで `cabocha-learn` を使って学習し直したモデルにはこの制限はない。
- configure の文字コードの既定は EUC-JP で、モデル (`/usr/local/lib/cabocha/model/*.model`) は make の中でテキスト形式からその文字コードで作られる。ここではドキュメントに従い、UTF-8 の IPA 辞書に合わせて `--with-charset=UTF8 --enable-utf8-only` を指定した。使うモデルや既定の設定は `/usr/local/etc/cabocharc` に書かれる。
- 最初の文では「太郎は」の係り先だけが違い、CaboCha は文末の「返した。」、GiNZA は「借りた」になった。主題の「は」は連体修飾節（ここでは「借りた本」の「借りた」）の中には入りにくいので、文末の述語に係るとする CaboCha の解析が普通の基準に合う。「借りた」の動作主も太郎であることは、係り受けではなく格解析や省略解析で扱う問題になる（この章の「Juman++ と KNP で係り受けと格関係を解析する」を参照）。
- 文節の単位も少し違う。「教えてあげた。」は CaboCha では 1 文節だが、GiNZA では「教えて」と「あげた。」の 2 文節になり、ほかの文節は「あげた。」や「教えて」に係る。GiNZA は単語単位の係り受け (UD) を解析してから文節をまとめていて、補助動詞の「あげる」が述語の中心 (ROOT) になるため。
- CaboCha は渡された文字列（1 行）を 1 文として扱う。「吾輩は猫である。名前はまだ無い。」をそのまま渡すと、「猫である。」が次の文の「無い。」に係ってしまう。文が複数あるときは、先に文に分割してから 1 文ずつ渡す。
- Python バインディングがビルドできない環境では、`subprocess.run(["cabocha", "-f1"], input=text, ...)` の出力を読めばよい。`*` で始まる行が文節の区切りで、2 列目が文節番号、3 列目が係り先（末尾の D を除く）になる。
- Python バインディングの `parse` が返す木はパーザの中のデータを指す。パーザを変数に入れずに `CaboCha.Parser("-n 1").parse(...)` と書くと、パーザが消えた後に木を読むことになり、カーネルが落ちた。
- 格子形式の最後の数値は係りやすさのスコアで、ドキュメントでは大きいほど係りやすいとされるが、実際の精度との関係は調べられていないとも書かれている。最初の文では、文末の「返した。」に係る「太郎は」「本を」「昨日」のスコアがどれも -1.474801 で、正しい係り先なのに負になった。文末の文節への係りは同じ値になることがあるので、スコアを確信度として使うのは避ける。